In [8]:
import cv2
import glob
import matplotlib.pyplot as plt
import numpy as np
from natsort import natsorted
import os
import sys
import pandas as pd
from scipy.ndimage import rotate, gaussian_filter

sys.path.append('../defect_functions') 
from defect_pairs import * 
from average_flows import * 

# image_list = glob.glob(r"C:\Users\victo\Downloads\SB_lab\BEER_DATA\ISF defects new\6\*.tif"); sigma=21
# image_list = glob.glob(r"C:\Users\victo\OneDrive - BGU\HBEC\s2(120-1057)\Raw\*.tif"); sigma=21
# image_list = glob.glob(r"D:\C2C12 and RPE in 6 well\15min5x\_1\Pos20\*.tif"); sigma=21
image_list = glob.glob(r"C:\Users\victo\OneDrive - BGU\BEER\star victor\cw\40x*.tif"); sigma=21
image_list = natsorted(image_list, key=lambda y: y.lower())

In [11]:
%matplotlib qt

frame = 10
fig, ax1 = plt.subplots(1,1,  figsize=(8,8))

ax1.clear(); ax1.axis('off') 


img = cv2.imread(image_list[frame])[:900,:900,0]

y, x = np.mgrid[0:img.shape[0], 0:img.shape[1]]

sigma = sigma
ori, plus, min = analyze_defects(img, sigma=sigma)

s = 21
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(6,6))
img_clahe = clahe.apply(img)
# ax1.imshow(255-img_clahe, cmap="gray")  
ax1.imshow(np.zeros_like(img, dtype=np.float32), cmap="gray")

nematic_plot(ax1, ori, plus, min, s=s)

# output_csv = r"C:\Users\victo\OneDrive - BGU\HBEC\s2(120-1057)\Nematics\output_Raw_frame160_500_sigma21.csv"; dist_thres=15; dframe=160
output_csv = r"D:\C2C12 and RPE in 6 well\15min5x\_1\Nematics\output_Pos20.csv"; dist_thres=15; dframe=0
all_pairs_df = pd.read_csv(output_csv).dropna(subset=["min_id","creation"]).dropna(subset=["min_id","creation"])

conditions = ((all_pairs_df.FRAME==frame-dframe)&(~all_pairs_df.creation))
ax1.scatter(
    all_pairs_df[conditions].xp, all_pairs_df[conditions].yp, 
    c='w', s=300, alpha=1., marker="^")


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\C2C12 and RPE in 6 well\\15min5x\\_1\\Nematics\\output_Pos20.csv'

In [3]:
img_h, img_w = cv2.imread(image_list[0]).shape[:2]
img_h, img_w

(1104, 1608)

In [15]:
from matplotlib.animation import FuncAnimation
%matplotlib qt

output_csv = r"C:\Users\victo\OneDrive - BGU\Dov\output_bacteria6_all.csv"; dist_thres=12
# output_csv = r"C:\Users\victo\OneDrive - BGU\HBEC\s2(120-1057)\Nematics\output_Raw_frame160_500_sigma21.csv"; dist_thres=21
# output_csv = r"D:\C2C12 and RPE in 6 well\15min5x\_1\Nematics\output_Pos20.csv"; dist_thres=15
all_pairs_df = pd.read_csv(output_csv).dropna(subset=["min_id","creation"]).dropna(subset=["min_id","creation"])
all_pairs_df_ = all_pairs_df.groupby(['plus_id','min_id']).filter(lambda x: len(x)>5).copy()

# img_h = cv2.imread(image_list[0]).shape[0]
img_h, img_w = cv2.imread(image_list[0]).shape[:2] # 1200, 1200
boundary = 30
conditions = (
    (all_pairs_df_.xp <= img_h-boundary) & (all_pairs_df_.xp >boundary) & 
    (all_pairs_df_.xm <= img_h-boundary) &(all_pairs_df_.xm >boundary) &
    (all_pairs_df_.yp <= img_h-boundary) & (all_pairs_df_.yp >boundary) & 
    (all_pairs_df_.ym <= img_h-boundary) & (all_pairs_df_.ym >boundary) 
    )
all_pairs_df__ = all_pairs_df_[conditions].copy()
plus_ids_to_remove = set(all_pairs_df_[~conditions].plus_id)
filt_pairs = all_pairs_df__[~all_pairs_df__.plus_id.isin(plus_ids_to_remove)]



# all_pairs_df_[all_pairs_df_.FRAME==1]

fig, ax1 = plt.subplots(1,1,  figsize=(8,8))
ax1.clear()
ax1.axis('off')

# Store defect data of each frame
imgs = []
oris = []
pluss = []
mins = []

ani_length = 100
sigma = 21
s = 31 #21
dframe = 0#160
process_type = 'fusion' #'fusion' #'creation'

for idx in range(ani_length):
    img = cv2.imread(image_list[dframe:][idx])[:img_h, :img_w, 0]
    imgs.append(img)

    ori, plus_df, min_df = analyze_defects(img, sigma=sigma)
    oris.append(ori)
    # pluss.append(plus_df)
    # mins.append(min_df)

y, x = np.mgrid[0:ori.shape[0], 0:ori.shape[1]]

def update(frame):
    ax1.clear()
    ax1.axis('off')
    ax1.imshow(255-imgs[frame], cmap="gray")
    # ax1.imshow(np.zeros_like(img, dtype=np.float32), cmap="gray")
    # nematic_plot(ax1, oris[frame], pluss[frame], mins[frame], s=s)

    ax1.quiver(x[::s,::s], y[::s,::s],
        np.cos(oris[frame])[::s,::s], np.sin(oris[frame])[::s,::s], 
        np.arctan2(np.sin(oris[frame]), np.cos(oris[frame]))[::s,::s],
        headaxislength=0, headwidth=0, headlength=0, width=.005,
        scale=55, pivot='mid', alpha=.3, cmap="hsv")   

    conditions = ((filt_pairs.FRAME==frame)&(filt_pairs[process_type]))
            # &(filt_pairs.yp<900)
    conditions_track = ((filt_pairs.FRAME<frame)&(filt_pairs.FRAME>frame-10)&(filt_pairs[process_type]))

    # Plot +1/2 defects
    # ax1.scatter(pluss[frame].x, pluss[frame].y, c='r', label='+1/2', s=50)
    ax1.scatter(
        filt_pairs[conditions].xp, filt_pairs[conditions].yp, 
        c='r', label='+1/2', s=120, alpha=0.6)
    ax1.scatter(
        filt_pairs[conditions_track].xp, 
        filt_pairs[conditions_track].yp, 
        c='r', s=60, alpha=.8, marker='.')
    # [ax1.text(dfi[2], dfi[3]-10, dfi[1], color='w', fontsize=12) for dfi in filt_pairs[conditions].values]

    # Plot -1/2 defects
    # ax1.scatter(mins[frame].x, mins[frame].y, c='b', label='-1/2', s=50)
    ax1.scatter(
        filt_pairs[conditions].xm, filt_pairs[conditions].ym, 
        c='b', label='-1/2', s=120, marker='^', alpha=0.6)
    ax1.scatter(
        filt_pairs[conditions_track].xm, 
        filt_pairs[conditions_track].ym, 
        c='b', s=60, alpha=.8, marker='.')    

    ax1.legend(loc='upper right')
    ax1.set_title(f"Frame {frame}")
    return ax1,

ani = FuncAnimation(fig, update, frames=range(ani_length), blit=False, repeat=True)
plt.show()

# ani.save(r'C:\Users\victo\Downloads\annihilation1.gif', writer='pillow')
# ani.save(r'C:\Users\victo\Downloads\cell_creation1.gif', writer='pillow')
ani.save(r"C:\Users\victo\Downloads\cell_" + process_type + ".gif", writer='pillow')

In [39]:
ani.save(r'C:\Users\victo\Downloads\cell_annihilation1.gif', writer='pillow')

In [31]:
frame = 6
conditions_track = ((filt_pairs.FRAME<frame)&(filt_pairs.FRAME>frame-10)&(~filt_pairs.creation))
filt_pairs[conditions_track]

,FRAME,plus_id,xp,yp,angp1,min_id,xm,ym,angm1,angm2,...,fuse_up,p_vel_angle,p_vel_angle_rel,anglep1_rel_vel_angle,fusion,creation,mp_angl1,mp_angl2,mp_angl3,mp_phase
3891,3,161,860.0,321.0,0.502655,413,821.0,467.0,6.220353,1.759292,...,False,3.141593,1.936853,1.812423,True,False,5.717699,1.256637,3.958407,1.549852
3892,4,161,850.0,321.0,0.753982,413,825.0,437.0,4.335398,6.157522,...,False,2.677945,2.060754,2.576156,True,False,3.581416,5.403539,1.068142,1.256637
3893,5,161,840.0,331.0,0.691150,413,825.0,432.0,4.272566,6.220353,...,False,2.356194,2.209710,2.899908,True,False,3.581416,5.529203,1.130973,1.319469
3954,3,402,386.0,242.0,5.403539,410,381.0,252.0,6.157522,2.073451,...,False,4.712389,5.588447,4.939892,True,False,0.753982,2.953097,5.152212,0.858702
3955,4,402,386.0,230.0,5.152212,410,369.0,250.0,4.209734,2.010619,...,False,3.663996,4.056693,5.496111,True,False,5.340708,3.141593,1.068142,1.089085
3956,5,402,353.0,223.0,5.529203,410,358.0,246.0,1.822124,4.209734,...,False,3.241261,3.377138,0.931206,True,False,2.576106,4.963716,0.628319,0.628319


In [20]:
frame = 10
conditions = (
        (all_pairs_df_.FRAME==frame)
        &(all_pairs_df_.xp<900)
        &(~all_pairs_df_.creation)
                )

for dfi in all_pairs_df_[conditions].values:
        print(dfi[1:4])
        break

[4 145.60260035763142 877.344285055576]


In [37]:
filtered_df = all_pairs_df_[(all_pairs_df_.xp <= 890) & (all_pairs_df_.xm <= 890)]
plus_ids_to_remove = set(all_pairs_df_[~((all_pairs_df_.xp <= 890) & (all_pairs_df_.xm <= 890))].plus_id)
filtered_df[~filtered_df.plus_id.isin(plus_ids_to_remove)].shape, all_pairs_df_[(all_pairs_df_.xp <= 890) & (all_pairs_df_.xm <= 890)].shape


((6736, 27), (6767, 27))

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
%matplotlib qt

# Create a figure and axis
fig, ax = plt.subplots()
xdata, ydata = [], []
ln, = plt.plot([], [], 'r-')

def init():
    ax.set_xlim(0, 2*np.pi)
    ax.set_ylim(-1.1, 1.1)
    return ln,

def update(frame):
    xdata.append(frame)
    ydata.append(np.sin(frame))
    ln.set_data(xdata, ydata)
    return ln,

# Create the animation
ani = FuncAnimation(fig, update, frames=np.linspace(0, 2*np.pi, 128),
                    init_func=init, blit=True)

# To display in a notebook
plt.show()

# To save the animation as a GIF
# ani.save('sine_wave.gif', writer='pillow')

In [1]:
import napari
viewer = napari.Viewer()
# viewer.add_image(data, name='astronaut')

ModuleNotFoundError: No module named 'napari'